# Final V3 English reproduction notebook

Copy this notebook into the extracted delivery directory before opening it. Requires KV260 PYNQ, Python 3.10+ and NumPy. Run one format/kernel at a time. Hardware switches default to False. This clean copy has not been physically re-executed; the original executed notebook is preserved under results/srcnn-board.

The deployment ZIP was frozen before physical validation, so its historical PENDING text is superseded by the audited evidence. Never edit hashed runtime files or mix BIT/HWH/model formats.


# SRCNN physical-board validation

Run on the KV260, from this delivery directory. All saved outputs start empty.

Do not run both formats at the same time. If a reset failure retains buffers, keep this kernel alive and reset/power-cycle the board before closing it.

In [ ]:
from pathlib import Path
import aero_board_test

DELIVERY = Path.cwd()
_, manifest, _, _ = aero_board_test.verify_delivery(DELIVERY)
print('Profile:', manifest['profile'])
print('No FPGA programming in this cell.')

In [ ]:
# Explicitly programs FPGA, but starts no inference or DMA.
RUN_PREFLIGHT = False
PREFLIGHT_OK = False
if RUN_PREFLIGHT:
    control_status = aero_board_test.preflight_board(DELIVERY)
    PREFLIGHT_OK = True
else:
    print('Set RUN_PREFLIGHT = True only when the board is ready.')

In [ ]:
# Set RUN_BOARD = True only when connected to the borrowed KV260.
RUN_BOARD = False

if RUN_BOARD:
    if not PREFLIGHT_OK:
        raise RuntimeError('Complete the control-path preflight first')
    from datetime import datetime, timezone
    output_dir = DELIVERY / ('board_results_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    report = aero_board_test.run_board(DELIVERY, output_dir, iterations=10, timeout=120)
    print('Status:', report['status'])
    print('Benchmark:', report['benchmark'])
    print('Saved:', output_dir)
else:
    print('Board execution disabled; set RUN_BOARD = True when ready.')

In [ ]:
import shutil

# Archive results only after a successful run.
if RUN_BOARD and report["status"] == "PASS":
    archive = shutil.make_archive(str(output_dir), "zip", root_dir=str(output_dir))
    print("Download results archive:", archive)
